# Modeles Avances — Fine-tuning de Transformers
## DistilBERT et RoBERTa pour la detection de fake news

**Objectif** : Utiliser des modeles de langage pre-entraines (embeddings contextuels) pour ameliorer la classification binaire de fake news.

**Approches** :
1. **DistilBERT** : Version compacte de BERT (40% plus petite, 60% plus rapide)
2. **RoBERTa** : Version optimisee de BERT (pre-entrainement plus robuste)

**Avantage par rapport aux baselines** :
- Representations contextuelles : le meme mot a des vecteurs differents selon le contexte
- Transfer learning : les modeles ont deja appris la structure du langage sur d'immenses corpus
- Fine-tuning : on adapte le modele a notre tache specifique

**Ref** :
- Devlin et al. (2019) — BERT
- Liu et al. (2019) — RoBERTa
- Sanh et al. (2019) — DistilBERT

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch : {torch.__version__}")

Device : cpu
PyTorch : 2.11.0+cpu


## 1. Chargement des donnees et preparation

In [3]:
df_train = pd.read_parquet(DATA_DIR / "liar_train.parquet")
df_valid = pd.read_parquet(DATA_DIR / "liar_valid.parquet")
df_test  = pd.read_parquet(DATA_DIR / "liar_test.parquet")

print(f"Train : {len(df_train):,} | Valid : {len(df_valid):,} | Test : {len(df_test):,}")

# On utilise le texte original (pas clean) pour les transformers qui ont leur propre tokenizer
train_texts = df_train["statement"].tolist()
train_labels = df_train["label_binary"].tolist()

valid_texts = df_valid["statement"].tolist()
valid_labels = df_valid["label_binary"].tolist()

test_texts = df_test["statement"].tolist()
test_labels = df_test["label_binary"].tolist()

Train : 10,240 | Valid : 1,284 | Test : 1,267


In [4]:
# Dataset PyTorch pour les transformers
class FakeNewsDataset(Dataset):
    """Dataset compatible avec HuggingFace Trainer."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Metriques pour le Trainer
def compute_metrics(eval_pred):
    """Calcule accuracy et F1 pour le Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

print("Dataset et metriques definis.")

Dataset et metriques definis.


## 2. Fine-tuning DistilBERT

**DistilBERT** (Sanh et al., 2019) est une version distillee de BERT :
- 6 couches au lieu de 12 → 40% plus petit
- 60% plus rapide a l'inference
- Conserve ~97% des performances de BERT

On fine-tune le modele `distilbert-base-uncased` sur notre tache de classification binaire.

In [5]:
# --- DistilBERT ---
MODEL_NAME_DB = "distilbert-base-uncased"
OUTPUT_DIR_DB = str(MODELS_DIR / "distilbert")

print(f"Chargement du tokenizer et modele : {MODEL_NAME_DB}")
tokenizer_db = AutoTokenizer.from_pretrained(MODEL_NAME_DB)
model_db = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DB, num_labels=2
)

# Preparation des datasets
train_dataset_db = FakeNewsDataset(train_texts, train_labels, tokenizer_db)
valid_dataset_db = FakeNewsDataset(valid_texts, valid_labels, tokenizer_db)
test_dataset_db  = FakeNewsDataset(test_texts, test_labels, tokenizer_db)

print(f"Datasets crees — Train: {len(train_dataset_db)}, Valid: {len(valid_dataset_db)}, Test: {len(test_dataset_db)}")

Chargement du tokenizer et modele : distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Datasets crees — Train: 10240, Valid: 1284, Test: 1267


In [ ]:
# Configuration de l'entrainement DistilBERT
training_args_db = TrainingArguments(
    output_dir=OUTPUT_DIR_DB,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),  # Mixed precision si GPU
)

trainer_db = Trainer(
    model=model_db,
    args=training_args_db,
    train_dataset=train_dataset_db,
    eval_dataset=valid_dataset_db,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Lancement du fine-tuning DistilBERT...")
train_result_db = trainer_db.train()
print(f"\nEntrainement termine — Loss finale : {train_result_db.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Lancement du fine-tuning DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.650927,0.656074,0.632399,0.625635


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Evaluation DistilBERT sur le test set
results_db = trainer_db.evaluate(test_dataset_db)
print(f"DistilBERT — Test Accuracy: {results_db['eval_accuracy']:.4f}  F1: {results_db['eval_f1_weighted']:.4f}")

# Predictions detaillees
preds_db = trainer_db.predict(test_dataset_db)
y_pred_db = np.argmax(preds_db.predictions, axis=-1)

print(f"\n{classification_report(test_labels, y_pred_db, target_names=['Fake', 'Real'])}")

# Sauvegarder le meilleur modele
trainer_db.save_model(str(MODELS_DIR / "distilbert_best"))
tokenizer_db.save_pretrained(str(MODELS_DIR / "distilbert_best"))
print(f"\nModele sauvegarde dans {MODELS_DIR / 'distilbert_best'}")

## 3. Fine-tuning RoBERTa

**RoBERTa** (Liu et al., 2019) ameliore BERT avec :
- Plus de donnees d'entrainement
- Suppression du NSP (Next Sentence Prediction)
- Sequences plus longues et plus grands batch sizes
- Dynamic masking au lieu de static masking

In [ ]:
# --- RoBERTa ---
MODEL_NAME_RB = "roberta-base"
OUTPUT_DIR_RB = str(MODELS_DIR / "roberta")

print(f"Chargement du tokenizer et modele : {MODEL_NAME_RB}")
tokenizer_rb = AutoTokenizer.from_pretrained(MODEL_NAME_RB)
model_rb = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_RB, num_labels=2
)

# Preparation des datasets
train_dataset_rb = FakeNewsDataset(train_texts, train_labels, tokenizer_rb)
valid_dataset_rb = FakeNewsDataset(valid_texts, valid_labels, tokenizer_rb)
test_dataset_rb  = FakeNewsDataset(test_texts, test_labels, tokenizer_rb)

# Configuration
training_args_rb = TrainingArguments(
    output_dir=OUTPUT_DIR_RB,
    num_train_epochs=5,
    per_device_train_batch_size=16,  # RoBERTa plus gros → batch plus petit
    per_device_eval_batch_size=32,
    learning_rate=1e-5,              # Learning rate plus faible pour RoBERTa
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer_rb = Trainer(
    model=model_rb,
    args=training_args_rb,
    train_dataset=train_dataset_rb,
    eval_dataset=valid_dataset_rb,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Lancement du fine-tuning RoBERTa...")
train_result_rb = trainer_rb.train()
print(f"\nEntrainement termine — Loss finale : {train_result_rb.training_loss:.4f}")

In [ ]:
# Evaluation RoBERTa sur le test set
results_rb = trainer_rb.evaluate(test_dataset_rb)
print(f"RoBERTa — Test Accuracy: {results_rb['eval_accuracy']:.4f}  F1: {results_rb['eval_f1_weighted']:.4f}")

preds_rb = trainer_rb.predict(test_dataset_rb)
y_pred_rb = np.argmax(preds_rb.predictions, axis=-1)

print(f"\n{classification_report(test_labels, y_pred_rb, target_names=['Fake', 'Real'])}")

# Sauvegarder
trainer_rb.save_model(str(MODELS_DIR / "roberta_best"))
tokenizer_rb.save_pretrained(str(MODELS_DIR / "roberta_best"))
print(f"\nModele sauvegarde dans {MODELS_DIR / 'roberta_best'}")

## 4. Comparaison des modeles avances vs baselines

In [ ]:
# Charger les metriques des baselines
baselines_path = MODELS_DIR / "baselines_metrics.json"
if baselines_path.exists():
    with open(baselines_path) as f:
        baselines = json.load(f)
else:
    baselines = {}
    print("Pas de metriques baselines trouvees. Executez d'abord Modeles_de_Base.ipynb")

# Combiner toutes les metriques
all_metrics = {}
for name, m in baselines.items():
    all_metrics[name] = {"Accuracy": m.get("test_acc", 0), "F1-Score": m.get("test_f1", 0)}

all_metrics["DistilBERT"] = {
    "Accuracy": round(results_db["eval_accuracy"], 4),
    "F1-Score": round(results_db["eval_f1_weighted"], 4),
}
all_metrics["RoBERTa"] = {
    "Accuracy": round(results_rb["eval_accuracy"], 4),
    "F1-Score": round(results_rb["eval_f1_weighted"], 4),
}

compare_df = pd.DataFrame(all_metrics).T.sort_values("F1-Score", ascending=False)
print("=== Comparaison complete des modeles ===")
compare_df

In [ ]:
# Visualisation comparative
fig = go.Figure()
models = compare_df.index.tolist()

fig.add_trace(go.Bar(name="Accuracy", x=models, y=compare_df["Accuracy"], marker_color="#3498db"))
fig.add_trace(go.Bar(name="F1-Score", x=models, y=compare_df["F1-Score"], marker_color="#e74c3c"))

fig.update_layout(
    barmode="group",
    title="Comparaison Baselines vs Transformers — Accuracy & F1",
    yaxis_title="Score", template="plotly_white", height=500,
    yaxis=dict(range=[0, 1]),
)
fig.show()

In [ ]:
# Courbes d'entrainement DistilBERT
log_db = trainer_db.state.log_history
train_losses = [x["loss"] for x in log_db if "loss" in x and "eval_loss" not in x]
eval_entries = [x for x in log_db if "eval_loss" in x]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss d'entrainement", "F1 sur validation"])

fig.add_trace(go.Scatter(
    y=train_losses, mode="lines+markers", name="Train Loss",
    line=dict(color="#e74c3c"),
), row=1, col=1)

if eval_entries:
    eval_f1 = [x.get("eval_f1_weighted", 0) for x in eval_entries]
    fig.add_trace(go.Scatter(
        y=eval_f1, mode="lines+markers", name="Valid F1",
        line=dict(color="#2ecc71"),
    ), row=1, col=2)

fig.update_layout(title="Courbes d'entrainement — DistilBERT", template="plotly_white", height=400)
fig.show()

## 5. Synthese

**Resultats attendus** :
- DistilBERT devrait atteindre ~63-67% d'accuracy (amelioration de 2-5% vs baselines)
- RoBERTa devrait etre le meilleur modele (~65-68%) grace a son pre-entrainement plus robuste

**Analyse** :
- Les transformers capturent le **contexte semantique** que TF-IDF ne peut pas saisir
- Le gain est modeste car le LIAR dataset est intrinsequement difficile (textes courts, ambiguite)
- L'**early stopping** evite l'overfitting sur ce petit dataset
- Le cout computationnel est significativement plus eleve que les baselines

**Limites** :
- Sans GPU, l'entrainement est lent (~30-60 min par modele sur CPU)
- Le fine-tuning sur ~10K exemples est sous-optimal pour des modeles aussi grands
- Le dataset LIAR contient des declarations qui necessitent des connaissances factuelles externes

**Prochaine etape** : Evaluation hors-domaine sur FakeNewsNet/BuzzFeed pour tester la generalisation.